<a href="https://colab.research.google.com/github/vzuza/codigos_amanha/blob/main/projeto_codigos_pessoal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
from google.colab import drive
from IPython.display import display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
import matplotlib as mpl
import geopandas as gpd
import zipfile
import os
import re
import unicodedata
import requests


In [19]:
#Caminho dos dados no drive (IBGE)

drive.mount('/content/drive')

PASTA_IBGE = '/content/drive/MyDrive/Codigos_Projeto/IBGE'

caminho_alfabetizacao = os.path.join(
    PASTA_IBGE, 'alfabetizacao_BR.xlsx'
)

caminho_domicilio1 = os.path.join(
    PASTA_IBGE, 'caracteristicas_domicilio1_BR.xlsx'
)

caminho_domicilio2 = os.path.join(
    PASTA_IBGE, 'caracteristicas_domicilio2_BR.xlsx'
)

caminho_demografia = os.path.join(
    PASTA_IBGE, 'demografia_BR.xlsx'
)

caminho_renda = os.path.join(
    PASTA_IBGE, 'renda_PR.csv'
)

arquivos = {
    'Alfabetização': caminho_alfabetizacao,
    'Domicílio 1': caminho_domicilio1,
    'Domicílio 2': caminho_domicilio2,
    'Demografia': caminho_demografia,
    'Renda': caminho_renda
}

df_alfabetizacao_BR = pd.read_excel(caminho_alfabetizacao)
df_domicilio1_BR = pd.read_excel(caminho_domicilio1)
df_domicilio2_BR = pd.read_excel(caminho_domicilio2)
df_demografia_BR = pd.read_excel(caminho_demografia)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
#API renda IBGE

url = (
    'https://apisidra.ibge.gov.br/values/'
    't/10295/n6/all/v/13534/p/2022/'
    'c2/6794/c86/95251/c58/95253'
)


resposta = requests.get(url, timeout=120)
resposta.raise_for_status()

dados = resposta.json()


df_renda_PR = pd.DataFrame(dados[1:]).rename(
    columns=dados[0]
)

print(df_renda_PR.columns.tolist())

col_codigo = 'Município (Código)'


df_renda_PR = df_renda_PR[
    df_renda_PR[col_codigo].astype(str).str.startswith('41')
].copy()


df_renda_PR.to_csv(
    'renda_municipios_PR.csv',
    index=False,
    encoding='utf-8-sig'
)


['Nível Territorial (Código)', 'Nível Territorial', 'Unidade de Medida (Código)', 'Unidade de Medida', 'Valor', 'Município (Código)', 'Município', 'Variável (Código)', 'Variável', 'Ano (Código)', 'Ano', 'Sexo (Código)', 'Sexo', 'Cor ou raça (Código)', 'Cor ou raça', 'Grupo de idade (Código)', 'Grupo de idade']


In [21]:
# Caminho dos dados no drive (S2ID)

PASTA_S2ID = '/content/drive/MyDrive/Codigos_Projeto/S2ID'

caminho_s2id = os.path.join(
    PASTA_S2ID, 'S2ID_PR_2015_2025.csv'
)

df_s2id = pd.read_csv(
    caminho_s2id,
    encoding='utf-8-sig'
)

print(f'A base de dados possui {df_s2id.shape[0]} linhas e {df_s2id.shape[1]} colunas')

A base de dados possui 1157 linhas e 55 colunas


In [22]:
#Limpando os dados do IBGE BR para IBGE PR

df_alfabetizacao_PR = df_alfabetizacao_BR[
    df_alfabetizacao_BR['CD_MUN'].astype(str).str.startswith('41')
].copy()
print(f'A base de dados possui {df_alfabetizacao_PR.shape[0]} linhas e {df_alfabetizacao_PR.shape[1]} colunas')

df_domicilio1_PR = df_domicilio1_BR[
    df_domicilio1_BR['CD_MUN'].astype(str).str.startswith('41')
].copy()

print(f'A base de dados possui {df_domicilio1_PR .shape[0]} linhas e {df_domicilio1_PR .shape[1]} colunas')

df_domicilio2_PR = df_domicilio2_BR[
    df_domicilio2_BR['CD_MUN'].astype(str).str.startswith('41')
].copy()
print(f'A base de dados possui {df_domicilio2_PR.shape[0]} linhas e {df_domicilio2_PR.shape[1]} colunas')

df_demografia_PR = df_demografia_BR[
    df_demografia_BR['CD_MUN'].astype(str).str.startswith('41')
].copy()
print(f'A base de dados possui {df_demografia_PR.shape[0]} linhas e {df_demografia_PR.shape[1]} colunas')

A base de dados possui 399 linhas e 364 colunas
A base de dados possui 399 linhas e 91 colunas
A base de dados possui 399 linhas e 408 colunas
A base de dados possui 399 linhas e 38 colunas


In [23]:
#Padronizando os dados de alfabetização (IBGE/PR)

df_alfabetizacao_limpo = df_alfabetizacao_PR[
    ['CD_MUN', 'NM_MUN', 'V00900', 'V00901']
].copy()

df_alfabetizacao_limpo = df_alfabetizacao_limpo.rename(columns={
    'CD_MUN': 'codigo_municipio',
    'NM_MUN': 'municipio',
    'V00900': 'populacao_alfabetizada_15mais',
    'V00901': 'populacao_nao_alfabetizada_15mais'
})

df_alfabetizacao_limpo.head()
print(f'A planilha de alfabetização possui {df_alfabetizacao_limpo.shape[0]} linhas e {df_alfabetizacao_limpo.shape[1]} colunas')


A planilha de alfabetização possui 399 linhas e 4 colunas


In [24]:
#Padronizando os dados de demografia (IBGE/PR)

df_demografia_limpo = df_demografia_PR[
    [
        'CD_MUN',
        'NM_MUN',
        'V01007',
        'V01008',
        'V01031',
        'V01032',
        'V01033',
        'V01040',
        'V01041'
    ]
].copy()

df_demografia_limpo = df_demografia_limpo.rename(columns={
    'CD_MUN': 'codigo_municipio',
    'NM_MUN': 'municipio',
    'V01007': 'populacao_masculina',
    'V01008': 'populacao_feminina',
    'V01031': 'populacao_0_4_anos',
    'V01032': 'populacao_5_9_anos',
    'V01033': 'populacao_10_14_anos',
    'V01040': 'populacao_60_69_anos',
    'V01041': 'populacao_70_mais_anos'
})

df_demografia_limpo.head()
print(f'A planilha de demografia possui {df_demografia_limpo.shape[0]} linhas e {df_demografia_limpo.shape[1]} colunas')


A planilha de demografia possui 399 linhas e 9 colunas


In [25]:
#Padronizando os dados das características de domicilio 1 e 2 / saneamento (IBGE/PR)

# características de domicilio 2

df_saneamento_limpo = df_domicilio2_PR[
    [
        'CD_MUN',
        'NM_MUN',
        'V00312',
        'V00313',
        'V00314',
        'V00315',
        'V00316'
    ]
].copy()

df_saneamento_limpo = df_saneamento_limpo.rename(columns={
    'CD_MUN': 'codigo_municipio',
    'NM_MUN': 'municipio',
    'V00312': 'domicilios_fossa_rudimentar',
    'V00313': 'domicilios_vala',
    'V00314': 'domicilios_rio_lago_mar',
    'V00315': 'domicilios_outra_forma',
    'V00316': 'domicilios_sem_banheiro_sanitario'
})

# características de domicilio 1

df_total_domicilios = df_domicilio1_PR[
    ['CD_MUN', 'V00001']
].copy()

df_total_domicilios = df_total_domicilios.rename(columns={
    'CD_MUN': 'codigo_municipio',
    'V00001': 'total_domicilios'
})


df_saneamento_limpo['codigo_municipio'] = (
    df_saneamento_limpo['codigo_municipio'].astype(str)
)

df_total_domicilios['codigo_municipio'] = (
    df_total_domicilios['codigo_municipio'].astype(str)
)


#juntando as tabelas

df_saneamento_limpo = df_saneamento_limpo.merge(
    df_total_domicilios,
    on='codigo_municipio',
    how='left',
    validate='one_to_one'
)

df_saneamento_limpo.head()
print(f'A planilha de saneamento possui {df_saneamento_limpo.shape[0]} linhas e {df_saneamento_limpo.shape[1]} colunas')

A planilha de saneamento possui 399 linhas e 8 colunas


In [26]:
#Padronizando os dados de renda (IBGE/PR)

df_renda_PR = df_renda_PR[
    ['Município (Código)', 'Município', 'Valor']
].copy()

df_renda_PR = df_renda_PR.rename(columns={
    'Município (Código)': 'COD',
    'Município': 'MUN',
    'Valor': 'RENDA_DOMICILIAR_PER_CAPITA'
})

df_renda_PR['COD'] = df_renda_PR['COD'].astype(str)

df_renda_PR['RENDA_DOMICILIAR_PER_CAPITA'] = pd.to_numeric(
    df_renda_PR['RENDA_DOMICILIAR_PER_CAPITA'],
    errors='coerce'
)

display(df_renda_PR.head())

,COD,MUN,RENDA_DOMICILIAR_PER_CAPITA
3912,4100103,Abatiá - PR,920.00
3913,4100202,Adrianópolis - PR,808.00
3914,4100301,Agudos do Sul - PR,1000.00
3915,4100400,Almirante Tamandaré - PR,1066.67
3916,4100459,Altamira do Paraná - PR,978.00


In [27]:
#Modelando os dados de analfabetismo IBGE

df_alfabetizacao_limpo['populacao_alfabetizada_15mais'] = pd.to_numeric(
    df_alfabetizacao_limpo['populacao_alfabetizada_15mais'],
    errors='coerce'
)

df_alfabetizacao_limpo['populacao_nao_alfabetizada_15mais'] = pd.to_numeric(
    df_alfabetizacao_limpo['populacao_nao_alfabetizada_15mais'],
    errors='coerce'
)

df_alfabetizacao_limpo['taxa_analfabetismo'] = (
    df_alfabetizacao_limpo['populacao_nao_alfabetizada_15mais'] /
    (
        df_alfabetizacao_limpo['populacao_alfabetizada_15mais'] +
        df_alfabetizacao_limpo['populacao_nao_alfabetizada_15mais']
    )
) * 100

df_alfabetizacao_limpo.head()
df_alfabetizacao_limpo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 399 entries, 3912 to 4310
Data columns (total 5 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   codigo_municipio                   399 non-null    int64  
 1   municipio                          399 non-null    object 
 2   populacao_alfabetizada_15mais      399 non-null    int64  
 3   populacao_nao_alfabetizada_15mais  399 non-null    int64  
 4   taxa_analfabetismo                 399 non-null    float64
dtypes: float64(1), int64(3), object(1)
memory usage: 18.7+ KB


In [28]:
#Modelando os dados de demografia (IBGE/PR)

colunas_numericas = [
    'populacao_masculina',
    'populacao_feminina',
    'populacao_0_4_anos',
    'populacao_5_9_anos',
    'populacao_10_14_anos',
    'populacao_60_69_anos',
    'populacao_70_mais_anos'
]

for coluna in colunas_numericas:
    df_demografia_limpo[coluna] = pd.to_numeric(
        df_demografia_limpo[coluna],
        errors='coerce'
    )

#População total

df_demografia_limpo['populacao_total'] = (
    df_demografia_limpo['populacao_masculina'] +
    df_demografia_limpo['populacao_feminina']
)

# % população de 0 a 14 anos

df_demografia_limpo['populacao_0_14_anos'] = (
    df_demografia_limpo['populacao_0_4_anos'] +
    df_demografia_limpo['populacao_5_9_anos'] +
    df_demografia_limpo['populacao_10_14_anos']
)

df_demografia_limpo['percentual_0_14_anos'] = (
    df_demografia_limpo['populacao_0_14_anos'] /
    df_demografia_limpo['populacao_total']
) * 100

# % população de 65 anos ou mais

df_demografia_limpo['populacao_60_mais_anos'] = (
    df_demografia_limpo['populacao_60_69_anos'] +
    df_demografia_limpo['populacao_70_mais_anos']
)

df_demografia_limpo['percentual_60_mais_anos'] = (
    df_demografia_limpo['populacao_60_mais_anos'] /
    df_demografia_limpo['populacao_total']
) * 100

df_demografia_limpo.head()

,codigo_municipio,municipio,populacao_masculina,populacao_feminina,populacao_0_4_anos,populacao_5_9_anos,populacao_10_14_anos,populacao_60_69_anos,populacao_70_mais_anos,populacao_total,populacao_0_14_anos,percentual_0_14_anos,populacao_60_mais_anos,percentual_60_mais_anos
3912,4100103,Abatiá,3599,3634,468,473,439,858,697,7233,1380,19.079220,1555,21.498687
3913,4100202,Adrianópolis,3139,3073,386,416,477,634,555,6212,1279,20.589182,1189,19.140373
3914,4100301,Agudos do Sul,5275,4958,704,755,725,1064,709,10233,2184,21.342715,1773,17.326297
3915,4100400,Almirante Tamandaré,59028,60797,8185,10003,8909,8511,4988,119825,27097,22.613812,13499,11.265596
3916,4100459,Altamira do Paraná,1818,1772,209,246,221,383,311,3590,676,18.830084,694,19.331476


In [29]:
#Modelando os dados de saneamento (IBGE/PR)

colunas_numericas = [
    'total_domicilios',
    'domicilios_fossa_rudimentar',
    'domicilios_vala',
    'domicilios_rio_lago_mar',
    'domicilios_outra_forma',
    'domicilios_sem_banheiro_sanitario'
]

for coluna in colunas_numericas:
    df_saneamento_limpo[coluna] = pd.to_numeric(
        df_saneamento_limpo[coluna],
        errors='coerce'
    )

df_saneamento_limpo['domicilios_saneamento_inadequado'] = (
    df_saneamento_limpo[
        [
            'domicilios_fossa_rudimentar',
            'domicilios_vala',
            'domicilios_rio_lago_mar',
            'domicilios_outra_forma',
            'domicilios_sem_banheiro_sanitario'
        ]
    ].sum(axis=1, min_count=5)
)

df_saneamento_limpo['percentual_saneamento_inadequado'] = (
    df_saneamento_limpo['domicilios_saneamento_inadequado'] /
    df_saneamento_limpo['total_domicilios'].replace(0, np.nan)
) * 100


df_saneamento_limpo.head()

,codigo_municipio,municipio,domicilios_fossa_rudimentar,domicilios_vala,domicilios_rio_lago_mar,domicilios_outra_forma,domicilios_sem_banheiro_sanitario,total_domicilios,domicilios_saneamento_inadequado,percentual_saneamento_inadequado
0,4100103,Abatiá,698,4,34,0,0,2711,736,27.148654
1,4100202,Adrianópolis,749,48,90,6,0,2304,893,38.758681
2,4100301,Agudos do Sul,2369,3,0,4,0,3633,2376,65.400495
3,4100400,Almirante Tamandaré,4873,167,365,40,0,40796,5445,13.346897
4,4100459,Altamira do Paraná,1089,0,0,0,0,1330,1089,81.879699


In [30]:
# Modelando os dados de renda (IBGE/PR)

renda_min = df_renda_PR['RENDA_DOMICILIAR_PER_CAPITA'].min()
renda_max = df_renda_PR['RENDA_DOMICILIAR_PER_CAPITA'].max()

df_renda_PR['VULNERABILIDADE_RENDA'] = (
    (renda_max - df_renda_PR['RENDA_DOMICILIAR_PER_CAPITA']) /
    (renda_max - renda_min)
)

In [31]:
# Padronização e integração dos códigos do IBGE (df_vulnerabilidade)

bases = [
    df_demografia_limpo,
    df_alfabetizacao_limpo,
    df_saneamento_limpo
]

for df in bases:
    df['codigo_municipio'] = (
        df['codigo_municipio']
        .astype(str)
        .str.replace(r'\.0$', '', regex=True)
        .str.zfill(7)
    )

df_renda_PR['COD'] = (
    df_renda_PR['COD']
    .astype(str)
    .str.zfill(7)
)


df_vulnerabilidade = df_demografia_limpo[
    ['codigo_municipio', 'municipio',
     'percentual_0_14_anos',
     'percentual_60_mais_anos']
].copy()


df_vulnerabilidade = df_vulnerabilidade.merge(
    df_alfabetizacao_limpo[
        ['codigo_municipio', 'taxa_analfabetismo']
    ],
    on='codigo_municipio',
    how='left',
    validate='one_to_one'
)

df_vulnerabilidade = df_vulnerabilidade.merge(
    df_saneamento_limpo[
        ['codigo_municipio',
         'percentual_saneamento_inadequado']
    ],
    on='codigo_municipio',
    how='left',
    validate='one_to_one'
)


df_vulnerabilidade = df_vulnerabilidade.merge(
    df_renda_PR[
        ['COD', 'RENDA_DOMICILIAR_PER_CAPITA']
    ],
    left_on='codigo_municipio',
    right_on='COD',
    how='left',
    validate='one_to_one'
)

df_vulnerabilidade.drop(columns='COD', inplace=True)

print('Dimensões:', df_vulnerabilidade.shape)
print('Municípios:', df_vulnerabilidade['codigo_municipio'].nunique())

display(df_vulnerabilidade.head())

Dimensões: (399, 7)
Municípios: 399


,codigo_municipio,municipio,percentual_0_14_anos,percentual_60_mais_anos,taxa_analfabetismo,percentual_saneamento_inadequado,RENDA_DOMICILIAR_PER_CAPITA
0,4100103,Abatiá,19.079220,21.498687,11.010429,27.148654,920.00
1,4100202,Adrianópolis,20.589182,19.140373,12.461758,38.758681,808.00
2,4100301,Agudos do Sul,21.342715,17.326297,6.689878,65.400495,1000.00
3,4100400,Almirante Tamandaré,22.613812,11.265596,3.490831,13.346897,1066.67
4,4100459,Altamira do Paraná,18.830084,19.331476,10.603981,81.879699,978.00


In [32]:
# Indicadores em que valores maiores significam

# maior vulnerabilidade

indicadores_diretos = [
    'percentual_0_14_anos',
    'percentual_60_mais_anos',
    'taxa_analfabetismo',
    'percentual_saneamento_inadequado'
]

for coluna in indicadores_diretos:

    minimo = df_vulnerabilidade[coluna].min()
    maximo = df_vulnerabilidade[coluna].max()

    df_vulnerabilidade[coluna + '_norm'] = (
        (df_vulnerabilidade[coluna] - minimo) /
        (maximo - minimo)
    )

# Renda: normalização inversa

minimo = df_vulnerabilidade[
    'RENDA_DOMICILIAR_PER_CAPITA'
].min()

maximo = df_vulnerabilidade[
    'RENDA_DOMICILIAR_PER_CAPITA'
].max()

df_vulnerabilidade['renda_norm'] = (
    (maximo - df_vulnerabilidade['RENDA_DOMICILIAR_PER_CAPITA']) /
    (maximo - minimo)
)

In [33]:
#Calcular o componente V

colunas_v = [
    'percentual_0_14_anos_norm',
    'percentual_60_mais_anos_norm',
    'renda_norm',
    'taxa_analfabetismo_norm',
    'percentual_saneamento_inadequado_norm'
]

df_vulnerabilidade['V'] = (
    df_vulnerabilidade[colunas_v].mean(axis=1)
)

display(
    df_vulnerabilidade[
        ['municipio', 'V']
    ].sort_values('V', ascending=False).head(20)
)

,municipio,V
339,São Jerônimo da Serra,0.735358
183,Laranjal,0.730283
307,Rio Branco do Ivaí,0.727577
237,Nova Santa Bárbara,0.723905
126,Grandes Rios,0.723213
105,Espigão Alto do Iguaçu,0.721642
216,Mato Rico,0.715936
395,Doutor Ulysses,0.697681
96,Diamante do Sul,0.696842
160,Itaúna do Sul,0.687533


In [36]:
#Padronizando os dados do S2ID

df_s2id_pr = df_s2id.copy()

df_s2id_pr['UF'] = (
    df_s2id_pr['UF'].astype(str).str.strip().str.upper()
)

df_s2id_pr['ANO_BASE'] = pd.to_numeric(
    df_s2id_pr['ANO_BASE'],
    errors='coerce'
)

df_s2id_pr = df_s2id_pr[
    (df_s2id_pr['UF'] == 'PR') &
    (df_s2id_pr['ANO_BASE'].between(2015, 2025))
].copy()

print('Registros:', len(df_s2id_pr))
print('Municípios:', df_s2id_pr['Município'].nunique())
print('Anos:', sorted(df_s2id_pr['ANO_BASE'].dropna().unique()))

Registros: 1157
Municípios: 321
Anos: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [37]:
print('Registros únicos:', df_s2id_pr['Registro'].nunique())

print(
    'Registros duplicados:',
    df_s2id_pr['Registro'].duplicated().sum()
)

print(
    'Protocolos únicos:',
    df_s2id_pr['Protocolo'].nunique()
)

duplicados = df_s2id_pr[
    df_s2id_pr['Registro'].duplicated(keep=False)
].sort_values('Registro')

display(duplicados.head(20))

Registros únicos: 602
Registros duplicados: 555
Protocolos únicos: 1157


,UF,Município,Registro,Protocolo,COBRADE,Status,População,DH_Mortos,DH_Feridos,DH_Enfermos,...,"PEPL_Distribuição de combustíveis, especialmente os de uso doméstico (R$)",PEPL_Segurança pública (R$),PEPL_Ensino (R$),PEPR_Agricultura (R$),PEPR_Pecuária (R$),PEPR_Indústria (R$),PEPR_Comércio (R$),PEPR_Serviços (R$),ANO_BASE,ARQUIVO_ORIGEM
178,PR,Apucarana,01/02/2016,PR-F-4101408-12200-20160111,12200 - Enxurradas,Reconhecido,120884,0,0,0,...,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,2016,dados-2016.xls
190,PR,Tomazina,01/02/2016,PR-F-4127809-12200-20160111,12200 - Enxurradas,Reconhecido,8788,0,0,0,...,0.0,0.0,120000.00,5840000.0,600000.0,400000.0,290000.0,300000.0,2016,dados-2016.xls
306,PR,Dois Vizinhos,01/03/2024,PR-F-4107207-15110-20240227,15110 - Doenças infecciosas virais,Reconhecido,36198,0,0,1363,...,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,2024,dados-2024.xls
316,PR,Lidianópolis,01/03/2024,PR-F-4113429-15110-20240215,15110 - Doenças infecciosas virais,Reconhecido,3972,0,0,318,...,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,2024,dados-2024.xls
313,PR,Boa Vista da Aparecida,01/03/2024,PR-F-4103057-15110-20240219,15110 - Doenças infecciosas virais,Reconhecido,7911,0,0,565,...,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,2024,dados-2024.xls
429,PR,Curitiba,01/04/2020,PR-F-4106902-15110-20200401,15110 - Doenças infecciosas virais,Reconhecido,1746896,0,0,87,...,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,2020,dados-2020.xls
454,PR,Ibiporã,01/04/2020,PR-F-4109807-15110-20200319,15110 - Doenças infecciosas virais,Reconhecido,48200,0,0,1,...,0.0,0.0,221527.92,0.0,0.0,0.0,0.0,0.0,2020,dados-2020.xls
627,PR,Coronel Vivida,01/06/2021,PR-F-4106506-13213-20210528,13213 - Tempestade Local/Convectiva - Granizo,Reconhecido,21737,0,0,0,...,0.0,0.0,16455.50,0.0,0.0,0.0,0.0,0.0,2021,dados-2021.xls
631,PR,Boa Esperança do Iguaçu,01/06/2021,PR-F-4103024-14110-20210524,14110 - Estiagem,Registro,2768,0,0,0,...,0.0,0.0,0.00,12250000.0,4300000.0,0.0,0.0,0.0,2021,dados-2021.xls
628,PR,Enéas Marques,01/06/2021,PR-F-4107405-13213-20210528,13213 - Tempestade Local/Convectiva - Granizo,Reconhecido,6101,0,1,0,...,0.0,0.0,15746.20,0.0,150000.0,300000.0,1380000.0,308000.0,2021,dados-2021.xls


In [40]:
# Frequência de desastres por município

df_frequencia = (
    df_s2id_pr.groupby('Município')['Protocolo']
    .nunique()
    .reset_index(name='total_ocorrencias')
)

df_frequencia = df_frequencia.sort_values(
    'total_ocorrencias',
    ascending=False
)

df_frequencia_anual = (
    df_s2id_pr.groupby(['Município', 'ANO_BASE'])['Protocolo']
    .nunique()
    .reset_index(name='total_ocorrencias')
)

display(df_frequencia_anual.head(20))

,Município,ANO_BASE,total_ocorrencias
0,Agudos do Sul,2018,1
1,Altamira do Paraná,2022,1
2,Altamira do Paraná,2023,1
3,Alto Paraná,2023,1
4,Alto Paraná,2024,1
5,Alto Paraná,2025,1
6,Alto Paraíso,2015,1
7,Alto Piquiri,2025,1
8,Altônia,2017,1
9,Altônia,2022,1


In [41]:
print('Total de ocorrências:', df_s2id_pr['Protocolo'].nunique())
print('Total de municípios:', df_frequencia['Município'].nunique())
print('Soma das ocorrências por município:',
      df_frequencia['total_ocorrencias'].sum())

Total de ocorrências: 1157
Total de municípios: 321
Soma das ocorrências por município: 1157


In [42]:
#Normalização da Frequência (F)

df_frequencia['frequencia_log'] = np.log1p(
    df_frequencia['total_ocorrencias']
)

f_min = df_frequencia['frequencia_log'].min()
f_max = df_frequencia['frequencia_log'].max()

df_frequencia['F'] = (
    (df_frequencia['frequencia_log'] - f_min) /
    (f_max - f_min)
)

In [43]:
# Extrair o código IBGE do protocolo
df_s2id_pr['codigo_municipio'] = (
    df_s2id_pr['Protocolo']
    .astype(str)
    .str.extract(r'^PR-[^-]+-(\d{7})')
)

print('Códigos não identificados:',
      df_s2id_pr['codigo_municipio'].isna().sum())

display(
    df_s2id_pr[
        ['Município', 'Protocolo', 'codigo_municipio']
    ].head()
)

Códigos não identificados: 0


,Município,Protocolo,codigo_municipio
0,Cruzeiro do Sul,PR-F-4106704-12200-20251215,4106704
1,Alto Paraná,PR-F-4100608-12200-20251214,4100608
2,Curitiba,PR-F-4106902-12300-20251214,4106902
3,Ribeirão do Pinhal,PR-F-4121901-12300-20251214,4121901
4,São Sebastião da Amoreira,PR-F-4126009-12200-20251214,4126009


In [44]:
#Frequencia IBGE

df_frequencia = (
    df_s2id_pr.groupby('codigo_municipio')
    .agg(
        municipio=('Município', 'first'),
        total_ocorrencias=('Protocolo', 'nunique')
    )
    .reset_index()
)

print('Municípios com ocorrências:', len(df_frequencia))
display(df_frequencia.head())

Municípios com ocorrências: 321


,codigo_municipio,municipio,total_ocorrencias
0,4100301,Agudos do Sul,1
1,4100459,Altamira do Paraná,2
2,4100509,Altônia,3
3,4100608,Alto Paraná,3
4,4100707,Alto Piquiri,1


In [45]:
# Padronizar os códigos IBGE

df_demografia_limpo['codigo_municipio'] = (
    df_demografia_limpo['codigo_municipio']
    .astype(str).str.zfill(7)
)

df_frequencia['codigo_municipio'] = (
    df_frequencia['codigo_municipio']
    .astype(str).str.zfill(7)
)

# Integrar a frequência aos 399 municípios

df_frequencia_completa = df_demografia_limpo[
    ['codigo_municipio', 'municipio']
].merge(
    df_frequencia[
        ['codigo_municipio', 'total_ocorrencias']
    ],
    on='codigo_municipio',
    how='left'
)

# Municípios sem ocorrência registrada recebem zero

df_frequencia_completa['total_ocorrencias'] = (
    df_frequencia_completa['total_ocorrencias']
    .fillna(0)
    .astype(int)
)

print('Total de municípios:', len(df_frequencia_completa))
print('Municípios com ocorrências:',
      (df_frequencia_completa['total_ocorrencias'] > 0).sum())
print('Municípios sem ocorrências:',
      (df_frequencia_completa['total_ocorrencias'] == 0).sum())

print('Total de ocorrências:',
      df_frequencia_completa['total_ocorrencias'].sum())

Total de municípios: 399
Municípios com ocorrências: 321
Municípios sem ocorrências: 78
Total de ocorrências: 1157


In [47]:
# Transformação logarítmica / Normalização entre 0 e 1

df_frequencia_completa['frequencia_log'] = np.log1p(
    df_frequencia_completa['total_ocorrencias']
)

f_min = df_frequencia_completa['frequencia_log'].min()
f_max = df_frequencia_completa['frequencia_log'].max()

df_frequencia_completa['F'] = (
    (df_frequencia_completa['frequencia_log'] - f_min) /
    (f_max - f_min)
)

display(
    df_frequencia_completa.sort_values(
        'F', ascending=False
    ).head(20)
)

,codigo_municipio,municipio,total_ocorrencias,frequencia_log,F
93,4106902,Curitiba,28,3.367296,1.000000
58,4104204,Campo Largo,14,2.708050,0.804221
89,4106605,Cruzeiro do Oeste,14,2.708050,0.804221
212,4115457,Marquinho,12,2.564949,0.761724
224,4116208,Morretes,11,2.484907,0.737953
309,4122305,Rio Negro,11,2.484907,0.737953
270,4119608,Pitanga,11,2.484907,0.737953
311,4122503,Roncador,10,2.397895,0.712113
68,4104808,Cascavel,10,2.397895,0.712113
290,4120903,Quedas do Iguaçu,10,2.397895,0.712113


In [48]:
#Indicador de impacto

# Danos humanos
colunas_DH = [
    'DH_Mortos',
    'DH_Feridos',
    'DH_Enfermos',
    'DH_Desabrigados',
    'DH_Desalojados',
    'DH_Desaparecidos',
    'DH_Outros Afetados'
]

# Danos materiais: valores monetários
colunas_DM = [
    col for col in df_s2id_pr.columns
    if col.startswith('DM_') and col.endswith('Valor')
]

# Prejuízos econômicos: setores públicos e privados
colunas_PE = [
    col for col in df_s2id_pr.columns
    if col.startswith(('PEPL_', 'PEPR_'))
]

print('Danos humanos:', colunas_DH)
print('\nDanos materiais:', colunas_DM)
print('\nPrejuízos econômicos:', colunas_PE)

print('\nQuantidade de variáveis:')
print('DH:', len(colunas_DH))
print('DM:', len(colunas_DM))
print('PE:', len(colunas_PE))

Danos humanos: ['DH_Mortos', 'DH_Feridos', 'DH_Enfermos', 'DH_Desabrigados', 'DH_Desalojados', 'DH_Desaparecidos', 'DH_Outros Afetados']

Danos materiais: ['DM_Unidades Habitacionais Valor', 'DM_Instalações públicas de saúde Valor', 'DM_Instalações públicas de ensino Valor', 'DM_Instalações públicas prestadoras de outros serviços Valor', 'DM_Instalações públicas de uso comunitário Valor', 'DM_Obras de infraestrutura pública Valor']

Prejuízos econômicos: ['PEPL_Assistência médica, saúde pública e atendimento de emergências médicas (R$)', 'PEPL_Abastecimento de água potável (R$)', 'PEPL_Esgoto de águas pluviais e sistema de esgotos sanitários (R$)', 'PEPL_Sistema de limpeza urbana e de recolhimento e destinação do lixo (R$)', 'PEPL_Sistema de desinfestação/desinfecção do habitat/controle de pragas e vetores (R$)', 'PEPL_Geração e distribuição de energia elétrica (R$)', 'PEPL_Telecomunicações (R$)', 'PEPL_Transportes locais, regionais e de longo curso (R$)', 'PEPL_Distribuição de combust

In [49]:
# Converter todas as variáveis de impacto
colunas_impacto = colunas_DH + colunas_DM + colunas_PE

for col in colunas_impacto:
    df_s2id_pr[col] = pd.to_numeric(
        df_s2id_pr[col],
        errors='coerce'
    )

# Verificar valores ausentes
print(
    df_s2id_pr[colunas_impacto]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

DH_Mortos                                                                                 0
DH_Feridos                                                                                0
DH_Enfermos                                                                               0
DH_Desabrigados                                                                           0
DH_Desalojados                                                                            0
DH_Desaparecidos                                                                          0
DH_Outros Afetados                                                                        0
DM_Unidades Habitacionais Valor                                                           0
DM_Instalações públicas de saúde Valor                                                    0
DM_Instalações públicas de ensino Valor                                                   0
DM_Instalações públicas prestadoras de outros serviços Valor                    

In [50]:
# Danos humanos por ocorrência
df_s2id_pr['DH_total'] = (
    df_s2id_pr[colunas_DH].sum(axis=1, min_count=1)
)

# Danos materiais por ocorrência (R$)
df_s2id_pr['DM_total'] = (
    df_s2id_pr[colunas_DM].sum(axis=1, min_count=1)
)

# Prejuízos econômicos por ocorrência (R$)
df_s2id_pr['PE_total'] = (
    df_s2id_pr[colunas_PE].sum(axis=1, min_count=1)
)

display(
    df_s2id_pr[
        ['Município', 'Protocolo', 'DH_total', 'DM_total', 'PE_total']
    ].head(10)
)

,Município,Protocolo,DH_total,DM_total,PE_total
0,Cruzeiro do Sul,PR-F-4106704-12200-20251215,250,243688.16,0.00
1,Alto Paraná,PR-F-4100608-12200-20251214,100,2100000.00,2100000.00
2,Curitiba,PR-F-4106902-12300-20251214,240,0.00,0.00
3,Ribeirão do Pinhal,PR-F-4121901-12300-20251214,3182,3160000.00,90000.00
4,São Sebastião da Amoreira,PR-F-4126009-12200-20251214,400,1069240.00,1684195.00
5,Santo Antônio da Platina,PR-F-4124103-12200-20251213,111,6103510.86,141036.00
6,Assaí,PR-F-4101903-12200-20251213,89,346500.00,160000.00
7,Bandeirantes,PR-F-4102406-12200-20251212,10436,417300.00,404650.00
8,Paranavaí,PR-F-4118402-13214-20251212,1420,18259570.17,22201845.92
9,Brasilândia do Sul,PR-F-4103370-12200-20251212,59,213000.00,213520.00


In [51]:
# Agregar impactos municipais
df_impacto = (
    df_s2id_pr.groupby('codigo_municipio')
    .agg(
        municipio=('Município', 'first'),
        DH_total=('DH_total', 'sum'),
        DM_total=('DM_total', 'sum'),
        PE_total=('PE_total', 'sum'),
        total_ocorrencias=('Protocolo', 'nunique')
    )
    .reset_index()
)

display(
    df_impacto.sort_values(
        'DH_total', ascending=False
    ).head(20)
)

,codigo_municipio,municipio,DH_total,DM_total,PE_total,total_ocorrencias
95,4108304,Foz do Iguaçu,197524,1.014924e+08,3.993162e+07,5
56,4104808,Cascavel,161343,8.500211e+10,2.057953e+09,10
212,4119608,Pitanga,113282,9.319533e+07,2.095645e+08,11
122,4110706,Irati,102032,5.654637e+06,8.982246e+07,6
224,4120606,Prudentópolis,94877,5.452008e+07,6.314415e+08,8
12,4101507,Arapongas,91245,1.140000e+07,1.560000e+07,1
310,4128104,Umuarama,77468,8.703862e+06,1.898563e+08,9
311,4128203,União da Vitória,61532,3.895788e+07,1.651973e+06,8
150,4113700,Londrina,53406,8.666898e+07,1.079252e+08,7
123,4110805,Iretama,49528,1.149152e+06,8.776248e+07,5


In [52]:
# Impactos modelagem dos dados S2ID

# Recuperar população municipal do IBGE
df_populacao = df_demografia_limpo[
    ['codigo_municipio', 'populacao_total']
].copy()

df_populacao['codigo_municipio'] = (
    df_populacao['codigo_municipio'].astype(str).str.zfill(7)
)

# Integrar os impactos aos 399 municípios
df_impacto_completo = df_populacao.merge(
    df_impacto[
        ['codigo_municipio', 'DH_total', 'DM_total', 'PE_total']
    ],
    on='codigo_municipio',
    how='left',
    validate='one_to_one'
)

# Municípios sem registros no S2ID recebem zero
colunas_impacto = ['DH_total', 'DM_total', 'PE_total']

df_impacto_completo[colunas_impacto] = (
    df_impacto_completo[colunas_impacto].fillna(0)
)

# Indicadores relativos à população municipal
df_impacto_completo['DH_per_capita'] = (
    df_impacto_completo['DH_total'] /
    df_impacto_completo['populacao_total']
) * 1000

df_impacto_completo['DM_per_capita'] = (
    df_impacto_completo['DM_total'] /
    df_impacto_completo['populacao_total']
)

df_impacto_completo['PE_per_capita'] = (
    df_impacto_completo['PE_total'] /
    df_impacto_completo['populacao_total']
)

display(df_impacto_completo.head())

print('Municípios:', len(df_impacto_completo))


,codigo_municipio,populacao_total,DH_total,DM_total,PE_total,DH_per_capita,DM_per_capita,PE_per_capita
0,4100103,7233,0.0,0.0,0.0,0.000000,0.000000,0.000000
1,4100202,6212,0.0,0.0,0.0,0.000000,0.000000,0.000000
2,4100301,10233,3550.0,150000.0,200000.0,346.916838,14.658458,19.544611
3,4100400,119825,0.0,0.0,0.0,0.000000,0.000000,0.000000
4,4100459,3590,339.0,100000.0,1014500.0,94.428969,27.855153,282.590529


Municípios: 399


In [53]:

#Normalização do impacto (I)

def normalizar_log(coluna):

    valores = np.log1p(coluna.clip(lower=0))

    minimo = valores.min()
    maximo = valores.max()

    if maximo == minimo:
        return pd.Series(0.0, index=coluna.index)

    return (valores - minimo) / (maximo - minimo)


# Normalização de cada dimensão
df_impacto_completo['DH_norm'] = normalizar_log(
    df_impacto_completo['DH_per_capita']
)

df_impacto_completo['DM_norm'] = normalizar_log(
    df_impacto_completo['DM_per_capita']
)

df_impacto_completo['PE_norm'] = normalizar_log(
    df_impacto_completo['PE_per_capita']
)


# Cálculo do componente Impacto
colunas_i = [
    'DH_norm',
    'DM_norm',
    'PE_norm'
]

df_impacto_completo['I'] = (
    df_impacto_completo[colunas_i].mean(axis=1)
)

display(
    df_impacto_completo[
        ['codigo_municipio', 'DH_norm',
         'DM_norm', 'PE_norm', 'I']
    ].sort_values('I', ascending=False).head(20)
)


,codigo_municipio,DH_norm,DM_norm,PE_norm,I
306,4122156,0.913620,0.729446,0.718054,0.787040
68,4104808,0.728016,1.000000,0.611082,0.779699
36,4103024,0.942122,0.646755,0.732219,0.773699
97,4107157,0.953566,0.629194,0.710183,0.764315
199,4114351,0.988635,0.584819,0.686180,0.753211
210,4115358,0.875141,0.608675,0.765652,0.749822
270,4119608,0.962404,0.638916,0.614578,0.738633
212,4115457,0.969172,0.582976,0.656373,0.736174
291,4121000,0.889719,0.556406,0.715010,0.720378
105,4107546,0.948537,0.497785,0.703691,0.716671


In [55]:
# integração dos componentes V + F + I

# 399 municípios
df_ippdn = df_vulnerabilidade[
    ['codigo_municipio', 'municipio', 'V']
].copy()

# FREQUENCIA
df_ippdn = df_ippdn.merge(
    df_frequencia_completa[
        ['codigo_municipio', 'F', 'total_ocorrencias']
    ],
    on='codigo_municipio',
    how='left',
    validate='one_to_one'
)

# IMPACTO
df_ippdn = df_ippdn.merge(
    df_impacto_completo[
        ['codigo_municipio', 'I']
    ],
    on='codigo_municipio',
    how='left',
    validate='one_to_one'
)

# Verificar valores ausentes
print('Valores ausentes:')
print(df_ippdn[['V', 'F', 'I']].isna().sum())

Valores ausentes:
V    0
F    0
I    0
dtype: int64


In [60]:

# ------------------ CÁLCULO DO IPPDN ----------------------


# Pesos iguais
peso_V = 1/3
peso_F = 1/3
peso_I = 1/3

df_ippdn['IPPDN'] = (
    peso_V * df_ippdn['V'] +
    peso_F * df_ippdn['F'] +
    peso_I * df_ippdn['I']
)

# Ordenar municípios por prioridade
df_ippdn = df_ippdn.sort_values(
    'IPPDN',
    ascending=False
).reset_index(drop=True)

# Ranking
df_ippdn['posicao'] = (
    df_ippdn.index + 1
)

display(df_ippdn.head(30))

print('Total de municípios:', len(df_ippdn))
print('Índice mínimo:', df_ippdn['IPPDN'].min())
print('Índice máximo:', df_ippdn['IPPDN'].max())
print('Média:', df_ippdn['IPPDN'].mean())


,codigo_municipio,municipio,V,F,total_ocorrencias,I,IPPDN,posicao,grau_prioridade
0,4115457,Marquinho,0.635214,0.761724,12,0.736174,0.711037,1,Alto
1,4107546,Espigão Alto do Iguaçu,0.721642,0.652519,8,0.716671,0.696944,2,Alto
2,4121000,Querência do Norte,0.636567,0.652519,8,0.720378,0.669821,3,Alto
3,4122503,Roncador,0.609908,0.712113,10,0.639976,0.653999,4,Alto
4,4114351,Manfrinópolis,0.587732,0.617540,7,0.753211,0.652828,5,Alto
5,4113254,Laranjal,0.730283,0.532106,5,0.684025,0.648805,6,Alto
6,4119608,Pitanga,0.467503,0.737953,11,0.738633,0.648030,7,Alto
7,4107157,Diamante D'Oeste,0.614650,0.532106,5,0.764315,0.637024,8,Alto
8,4106456,Coronel Domingos Soares,0.662312,0.577885,6,0.666571,0.635589,9,Alto
9,4103156,Bom Jesus do Sul,0.630388,0.577885,6,0.687390,0.631888,10,Alto


Total de municípios: 399
Índice mínimo: 0.052589023903291246
Índice máximo: 0.7110372367426088
Média: 0.4053147904209351


In [59]:

# ------------------- CLASSIFICAÇÃO DOS GRAUS DE PRIORIDADE -----------------

limites = [
    -0.000001,
    0.20,
    0.40,
    0.60,
    0.80,
    1.000001
]

classes = [
    'Muito baixo',
    'Baixo',
    'Moderado',
    'Alto',
    'Muito alto'
]

df_ippdn['grau_prioridade'] = pd.cut(
    df_ippdn['IPPDN'],
    bins=limites,
    labels=classes,
    include_lowest=True
)

# Quantidade de municípios por classe
print(
    df_ippdn['grau_prioridade']
    .value_counts()
    .sort_index()
)

grau_prioridade
Muito baixo     60
Baixo          105
Moderado       210
Alto            24
Muito alto       0
Name: count, dtype: int64


In [58]:
df_ippdn.to_excel(
    'IPPDN_PR_resultados.xlsx',
    index=False
)